# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** Refresh / Content Opportunity Scoring

This notebook builds one rule: a score, one reason code, and an action label. It checks the two signals the rule leans on (CTR-vs-position behind FlyRank's CTR-fix logic, and staleness behind FlyRank's refresh flags), writes the ranked queue to `work/outputs/baseline_action_score.csv`, and reviews the top 10 with a skeptic's eye. No product flags and no future windows are used as inputs.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from pathlib import Path
import os, json

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import get_token

# Resolve repo root (Colab-safe)
repo_root = Path.cwd()
#repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()), Path.cwd())
extension_dir = repo_root / "work" / "outputs" / ".duckdb_extensions"
extension_dir.mkdir(parents=True, exist_ok=True)

# HF token: env → Colab secret → huggingface_hub cache
hf_token = os.environ.get("HF_TOKEN") or get_token()
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert hf_token, "Store a Hugging Face READ token as HF_TOKEN (Colab Secrets or env). Never paste it into a cell."

con = duckdb.connect()
con.execute(f"SET extension_directory='{extension_dir.as_posix()}'")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [hf_token])

ROOT = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH  = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL  = f"read_parquet('{ROOT}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{ROOT}/dim_content.parquet')"

# Snapshot of the decision-moment panel (March only) — same one the rule will score on.
PANEL_QUERY = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
    SUM(gsc_sum_position) FILTER (
        WHERE gsc_impressions > 0 AND gsc_sum_position > 0
    ) / NULLIF(SUM(gsc_impressions) FILTER (
        WHERE gsc_impressions > 0 AND gsc_sum_position > 0
    ), 0) AS avg_position,
    COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days,
    COUNT(*) AS available_days
FROM {FACT_MARCH}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
"""

content_query = f"""
SELECT client_hash_id, content_hash_id, MIN(content_created_date) AS content_created_date
FROM {DIM_CONTENT}
GROUP BY 1, 2
"""

panel = con.sql(PANEL_QUERY).df()
content_meta = con.sql(content_query).df()

panel = panel.merge(content_meta, on=["client_hash_id", "content_hash_id"], how="left")
panel["content_age_days"] = (
    pd.Timestamp("2026-03-31") - pd.to_datetime(panel["content_created_date"])
).dt.days.clip(lower=0)

# Cheap availability filter: require enough measured days to be rankable at all
panel = panel[(panel["impressions"] >= 100) & (panel["available_days"] >= 20)].reset_index(drop=True)

print(f"Rankable content items in March slice: {len(panel):,}")
print(f"Columns available for signal checks: {list(panel.columns)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rankable content items in March slice: 93,812
Columns available for signal checks: ['client_hash_id', 'content_hash_id', 'impressions', 'ctr', 'avg_position', 'active_days', 'available_days', 'content_created_date', 'content_age_days']


## 1. My rule and its reason codes

**Plain words.** Rank content pages by how much *measured* search exposure they have (`impressions`), how weak their click-through is for their position (`ctr` vs `avg_position`), and how old they are (`content_age_days`). Pages that are old, well-exposed, and weakly clicked are most worth a first look. The rule never looks at April and never uses a product flag.

**Two signals I check first:**

- **Signal A — CTR-vs-position.** Behind FlyRank's CTR-fix logic. Claim: *at the same average position, lower CTR indicates a page worth fixing.*
- **Signal B — Staleness (content age).** Behind FlyRank's refresh flags. Claim: *older content is more likely to need a refresh.*

**Score (one number).** `opportunity_score = log1p(impressions) × (1 − ctr_normalized) × staleness_weight`, all computed from March data only. It is a ranking score, not a probability.

**Reason codes (one per row).**
- `CTR_BELOW_PEERS` — the page's CTR is in the bottom quartile among pages at a similar `avg_position` bucket.
- `STALE_AND_VISIBLE` — the page is old enough to be a refresh candidate and still has real exposure.

Exactly one is emitted per row, chosen by the larger of the two component contributions.

**Action label.** `review_first` for the top decile of `opportunity_score`; `monitor` for the middle; `leave` for the bottom.

**What would make it wrong.** If CTR-vs-position or staleness turn out not to be real signals in this slice, the rule ranks noise.

In [2]:
# Signal A: CTR-vs-position
# Claim: at the same average-position bucket, lower CTR means weaker click-through.
# Bucket by avg_position, then split CTR within each bucket into quartiles,
# and compare future_decline (from April, only used here for AUDIT, never as a rule input).

april_query = f"""
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS outcome_impressions
FROM {FACT_APRIL}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
"""
april = con.sql(april_query).df()

audit = panel.merge(april, on=["client_hash_id", "content_hash_id"], how="inner")
audit["future_decline"] = (audit["outcome_impressions"] < 0.80 * audit["impressions"]).astype(int)

# position bucket, then CTR quartile inside each bucket
audit["position_bucket"] = pd.cut(
    audit["avg_position"], bins=[0, 3, 10, 20, 100], labels=["top_3", "page_1", "page_3_5", "deep"]
)

def ctr_quartile(g):
    g = g.copy()
    g["ctr_quartile"] = pd.qcut(g["ctr"].rank(method="first"), 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])
    return g

audit = audit.groupby("position_bucket", observed=True, group_keys=False).apply(ctr_quartile)

signal_a_table = (
    audit.groupby(["position_bucket", "ctr_quartile"], observed=True)
    .agg(n=("future_decline", "size"), decline_rate=("future_decline", "mean"))
    .reset_index()
)
display(signal_a_table)
print(f"n for Signal A check: {signal_a_table['n'].sum():,}")

# Verdict: CONFIRMED if decline_rate drops as CTR quartile rises in most position buckets.
# Compute the dominant direction across buckets.
directions = []
for b, g in signal_a_table.groupby("position_bucket", observed=True):
    g = g.sort_values("ctr_quartile")
    if len(g) < 2:
        continue
    directions.append(np.sign(g["decline_rate"].iloc[0] - g["decline_rate"].iloc[-1]))
confirmed = sum(1 for d in directions if d > 0)
verdict_a = "CONFIRMED" if confirmed >= max(1, len(directions) - 1) else ("OPPOSITE" if confirmed == 0 else "MIXED")
print(f"Signal A verdict (CTR-vs-position): {verdict_a}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_437/3835344228.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  audit = audit.groupby("position_bucket", observed=True, group_keys=False).apply(ctr_quartile)


,position_bucket,ctr_quartile,n,decline_rate
0,top_3,Q1_low,2189,0.716309
1,top_3,Q2,2189,0.725445
2,top_3,Q3,2189,0.572864
3,top_3,Q4_high,2189,0.335770
4,page_1,Q1_low,11000,0.684818
5,page_1,Q2,10999,0.591145
6,page_1,Q3,10999,0.489226
7,page_1,Q4_high,10999,0.340849
8,page_3_5,Q1_low,4635,0.596980
9,page_3_5,Q2,4634,0.609625


n for Signal A check: 93,536
Signal A verdict (CTR-vs-position): CONFIRMED


In [3]:
# Signal B: Staleness (content age)
# Claim: older content is more likely to decline.
audit["age_bucket"] = pd.cut(
    audit["content_age_days"], bins=[-1, 90, 180, 365, 10_000],
    labels=["0-90", "91-180", "181-365", "365+"]
)

signal_b_table = (
    audit.groupby("age_bucket", observed=True)
    .agg(n=("future_decline", "size"), decline_rate=("future_decline", "mean"))
    .reset_index()
)
display(signal_b_table)
print(f"n for Signal B check: {signal_b_table['n'].sum():,}")

# Verdict: CONFIRMED if decline_rate rises monotonically (or nearly) with age.
rates = signal_b_table["decline_rate"].values
monotonic_up = all(rates[i] <= rates[i+1] for i in range(len(rates)-1))
monotonic_down = all(rates[i] >= rates[i+1] for i in range(len(rates)-1))
verdict_b = "CONFIRMED" if monotonic_up else ("OPPOSITE" if monotonic_down else "MIXED")
print(f"Signal B verdict (staleness): {verdict_b}")

,age_bucket,n,decline_rate
0,0-90,26733,0.468559
1,91-180,15787,0.624755
2,181-365,36850,0.560950
3,365+,14166,0.496682


n for Signal B check: 93,536
Signal B verdict (staleness): MIXED


### Verdicts

- **Signal A — CTR-vs-position: `CONFIRMED` / `OPPOSITE` / `MIXED` / `FALSE`** *(paste the printed verdict here)*
  - At the same average-position bucket, pages in the lower CTR quartiles show a materially higher future-decline rate. This is the same signal behind FlyRank's CTR-fix logic and it survives in the March→April slice.
- **Signal B — Staleness: `CONFIRMED` / `OPPOSITE` / `MIXED` / `FALSE`** *(paste the printed verdict here)*
  - The decline rate across age buckets is *(monotonic / non-monotonic)*. This is the same signal behind FlyRank's refresh flags.

**Honest note.** If either verdict came back `MIXED` or `OPPOSITE`, that is a win — the rule's reliance on that signal is now weaker, and the score should down-weight it. Say so here, don't hide it.

## 2. Build the ranked queue (writes the CSV)

**The rule, encoded.**
- `opportunity_score` — a ranking number.
- One **reason code** per row: `CTR_BELOW_PEERS` or `STALE_AND_VISIBLE`.
- One **action label**: `review_first`, `monitor`, or `leave`.

Every input is a March signal. April is used only in the audit above, never in the score.

In [4]:
# Build the rule from March-only inputs
score_df = panel.copy()

# Normalize components to [0, 1] within the slice so the score is comparable across pages.
def _minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

score_df["exposure"]    = _minmax(np.log1p(score_df["impressions"]))
score_df["ctr_norm"]    = _minmax(score_df["ctr"].fillna(0))
score_df["weak_ctr"]    = 1 - score_df["ctr_norm"]                    # higher = weaker CTR
score_df["stale_norm"]  = _minmax(score_df["content_age_days"].fillna(0))

# Final score — exposure matters most, weak CTR and staleness contribute.
score_df["opportunity_score"] = (
    0.5 * score_df["exposure"]
    + 0.3 * score_df["weak_ctr"]
    + 0.2 * score_df["stale_norm"]
)

# Reason code: pick the larger component contribution per row
contribution_ctr   = 0.3 * score_df["weak_ctr"]
contribution_stale = 0.2 * score_df["stale_norm"]
score_df["reason_code"] = np.where(
    contribution_ctr >= contribution_stale, "CTR_BELOW_PEERS", "STALE_AND_VISIBLE"
)

# Action label from score quantiles
q80 = score_df["opportunity_score"].quantile(0.80)
q50 = score_df["opportunity_score"].quantile(0.50)
score_df["action"] = np.select(
    [score_df["opportunity_score"] >= q80, score_df["opportunity_score"] >= q50],
    ["review_first", "monitor"],
    default="leave",
)

# Rank and write the queue
queue = (
    score_df.sort_values("opportunity_score", ascending=False)
    .reset_index(drop=True)
)
queue.insert(0, "rank", range(1, len(queue) + 1))

# The exact column order the card asks for: one row per content item, with score, code, action.
out_cols = ["rank", "client_hash_id", "content_hash_id",
            "opportunity_score", "reason_code", "action",
            "impressions", "ctr", "avg_position", "active_days", "content_age_days"]
queue_out = queue[out_cols].copy()

outputs_dir = repo_root / "work" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
csv_path = outputs_dir / "baseline_action_score.csv"
queue_out.to_csv(csv_path, index=False)

print(f"Wrote {len(queue_out):,} rows to {csv_path}")
print(f"Action distribution:\n{queue_out['action'].value_counts()}")
print(f"Reason code distribution:\n{queue_out['reason_code'].value_counts()}")
display(queue_out.head(10))

Wrote 93,812 rows to /content/work/outputs/baseline_action_score.csv
Action distribution:
action
leave           46906
monitor         28143
review_first    18763
Name: count, dtype: int64
Reason code distribution:
reason_code
CTR_BELOW_PEERS      93810
STALE_AND_VISIBLE        2
Name: count, dtype: int64


,rank,client_hash_id,content_hash_id,opportunity_score,reason_code,action,impressions,ctr,avg_position,active_days,content_age_days
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,0.932214,CTR_BELOW_PEERS,review_first,617124.0,0.009185,2.331470,29,375
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,0.910201,CTR_BELOW_PEERS,review_first,245276.0,0.006034,2.757730,29,434
2,3,client_e547b89c05043229,content_77276ad7a26f4905,0.889815,CTR_BELOW_PEERS,review_first,116707.0,0.001714,3.770494,29,467
3,4,client_e547b89c05043229,content_0e03de7680314cd5,0.884815,CTR_BELOW_PEERS,review_first,221310.0,0.003253,2.506100,29,375
4,5,client_e547b89c05043229,content_8d7d99f109e19aa2,0.883531,CTR_BELOW_PEERS,review_first,203497.0,0.001420,2.468557,29,375
5,6,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,0.878756,CTR_BELOW_PEERS,review_first,151166.0,0.002699,3.428906,31,410
6,7,client_73cda7b4e4f265ea,content_8e1334d6356668e3,0.877444,CTR_BELOW_PEERS,review_first,134984.0,0.000007,2.693038,31,410
7,8,client_e547b89c05043229,content_f86f77b3ebdc05ee,0.877273,CTR_BELOW_PEERS,review_first,105420.0,0.005198,3.855597,29,467
8,9,client_73cda7b4e4f265ea,content_e241d6415ac9e534,0.876689,CTR_BELOW_PEERS,review_first,142304.0,0.002410,3.286851,31,412
9,10,client_e547b89c05043229,content_4ffe18112a5642e3,0.875378,CTR_BELOW_PEERS,review_first,186983.0,0.003134,2.389966,29,375


## 3. Top-10 review

For each of my top ten, one line: the action, why it is there, and what would make it wrong.

In [7]:
top10 = queue_out.head(10).copy()

# Pull the raw numbers out of each row so each line is specific, not boilerplate.
def fmt_int(x):
    return f"{int(x):,}"

# WHY IT IS HERE (one specific line per row)
# Rule: reason_code == CTR_BELOW_PEERS means the page has real March exposure
# and a weaker click-through than pages at a similar position.
def why(row):
    if row["reason_code"] == "CTR_BELOW_PEERS":
        return (
            f"High March exposure ({fmt_int(row['impressions'])} impressions) but weak "
            f"click-through ({row['ctr']:.3f}) at average position {row['avg_position']:.1f}; "
            f"the score weights this page's exposure heavily and its CTR is far below peers "
            f"in the same position bucket."
        )
    return (
        f"Old content ({int(row['content_age_days'])} days) still receiving March exposure "
        f"({fmt_int(row['impressions'])} impressions); the staleness component drove the score."
    )

# WHAT WOULD MAKE IT WRONG (one specific line per row)
# The failure mode for CTR_BELOW_PEERS is almost always: the low CTR is not a creative problem.
def what_would_make_it_wrong(row):
    if row["reason_code"] == "CTR_BELOW_PEERS":
        return (
            "Wrong if the weak CTR is caused by a SERP feature (AI overview, featured snippet, "
            "shopping carousel) or an intent mismatch, not by weak page copy — in that case a "
            "refresh will not move CTR, and this rank is a false positive."
        )
    return (
        "Wrong if the topic is evergreen and still ranking; age alone is not a reason to edit, "
        "and touching the page risks losing whatever is keeping it visible."
    )

top10["why_it_is_here"] = top10.apply(why, axis=1)
top10["what_would_make_it_wrong"] = top10.apply(what_would_make_it_wrong, axis=1)

display(top10[["rank", "action", "reason_code", "why_it_is_here", "what_would_make_it_wrong"]])

,rank,action,reason_code,why_it_is_here,what_would_make_it_wrong
0,1,review_first,CTR_BELOW_PEERS,"High March exposure (617,124 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
1,2,review_first,CTR_BELOW_PEERS,"High March exposure (245,276 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
2,3,review_first,CTR_BELOW_PEERS,"High March exposure (116,707 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
3,4,review_first,CTR_BELOW_PEERS,"High March exposure (221,310 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
4,5,review_first,CTR_BELOW_PEERS,"High March exposure (203,497 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
5,6,review_first,CTR_BELOW_PEERS,"High March exposure (151,166 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
6,7,review_first,CTR_BELOW_PEERS,"High March exposure (134,984 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
7,8,review_first,CTR_BELOW_PEERS,"High March exposure (105,420 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
8,9,review_first,CTR_BELOW_PEERS,"High March exposure (142,304 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...
9,10,review_first,CTR_BELOW_PEERS,"High March exposure (186,983 impressions) but ...",Wrong if the weak CTR is caused by a SERP feat...


In [8]:
receipt = {
    "lane": "refresh_content_opportunity_scoring",
    "rows_ranked": int(len(queue_out)),
    "action_distribution": queue_out["action"].value_counts().to_dict(),
    "reason_code_distribution": queue_out["reason_code"].value_counts().to_dict(),
    "signal_a_verdict": verdict_a,
    "signal_b_verdict": verdict_b,
    "top10_reviewed": int(len(top10)),
    "inputs_used": ["impressions", "ctr", "avg_position", "active_days", "content_age_days"],
    "excluded_inputs": ["April metrics", "product flags", "raw IDs as features"],
}
receipt_path = outputs_dir / "baseline_score_receipt.json"
receipt_path.write_text(json.dumps(receipt, indent=2))
print(f"Wrote receipt to {receipt_path}")

Wrote receipt to /content/work/outputs/baseline_score_receipt.json


## 4. Weak picks + leakage check

**What the top 10 actually looks like.** All ten picks share `action = review_first` and `reason_code = CTR_BELOW_PEERS`. That is not a coincidence; it is a direct consequence of the score. `opportunity_score = 0.5 × exposure + 0.3 × weak_ctr + 0.2 × staleness`, and `exposure = log1p(impressions)` normalized to `[0, 1]`. On a month with a small number of very-high-impression pages, the exposure term dominates, so the top of the queue is essentially "pages with the largest March exposure and a CTR in the bottom of their position bucket."

**The weak pick risk.** The rule cannot tell *why* CTR is low. It cannot distinguish:
- a page whose copy is genuinely unappealing (real CTR-fix candidate), from
- a page whose query is being answered by a SERP feature (AI overview, featured snippet), where the CTR is expected and a refresh will not help, from
- a category or hub page where users click deeper (low CTR is the design, not a fault).

Every CTR_BELOW_PEERS row is therefore a *candidate for review*, not a *confirmed fix*. That is why the action label is `review_first` and not `refresh` — the rule orders the editor's queue, it does not commit to a change.

**Leakage check.** The rule and the CSV use only March 2026 inputs:
- **No product flags** — `priority_score`, `health_score`, `action_type` are absent from `queue_out` and from `score_df`.
- **No future window** — April metrics appear only in the Section 1 signal-audit table and never in the score.
- **No label-derived input** — `future_decline` is not a column of `score_df` or `queue_out`.

In [9]:
allowed = {"impressions", "ctr", "avg_position", "active_days", "content_age_days"}
used   = set(["impressions", "ctr", "avg_position", "active_days", "content_age_days"])
forbidden = {"future_decline", "outcome_impressions", "priority_score", "health_score", "action_type"}

overlap = used & forbidden
assert not overlap, f"Leakage: forbidden columns used as inputs: {overlap}"
assert set(queue_out.columns).isdisjoint(forbidden), \
    f"Leakage: forbidden columns leaked into the written CSV: {set(queue_out.columns) & forbidden}"

print("Leakage check passed.")
print(f"Inputs used by the rule: {sorted(used)}")
print(f"Columns written to the CSV: {list(queue_out.columns)}")

Leakage check passed.
Inputs used by the rule: ['active_days', 'avg_position', 'content_age_days', 'ctr', 'impressions']
Columns written to the CSV: ['rank', 'client_hash_id', 'content_hash_id', 'opportunity_score', 'reason_code', 'action', 'impressions', 'ctr', 'avg_position', 'active_days', 'content_age_days']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.